# 04 — Statistical Analysis and Visualization (multi-seed benchmark)

Final analysis: 360 runs (3 models × 4 optimizers × 30 seeds).

In [4]:
import os, json, warnings
import pathlib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from scipy import stats
from itertools import combinations
try:
    import scikit_posthocs as sp
    HAS_SP = True
except ImportError:
    HAS_SP = False
    print("scikit_posthocs no disponible, se omitirá Nemenyi CD diagram")
warnings.filterwarnings("ignore")
print("Imports OK")


Imports OK


In [ ]:
_candidates = ["/home/jovyan/work", "/home/sergioperez/solar-benchmark"]
BASE_DIR = str(next(p for p in _candidates if pathlib.Path(p).exists()))

OUTPUT_DIR  = os.path.join(BASE_DIR, "outputs")
PVGIS_DIR   = os.path.join(OUTPUT_DIR, "pvgis")
RESULTS_DIR = os.path.join(PVGIS_DIR, "benchmark", "results")
FIGURES_DIR = os.path.join(OUTPUT_DIR, "figures")
TABLES_DIR  = os.path.join(OUTPUT_DIR, "tables")

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)

PALETTE = {"PSO": "#0077BB", "GA": "#EE7733", "ABC": "#009988", "TPE": "#CC3311"}
MODELS  = ["MLP", "LSTM", "GRU"]
OPTS    = ["PSO", "GA", "ABC", "TPE"]

print(f"BASE_DIR: {BASE_DIR}")
print(f"RESULTS_DIR: {RESULTS_DIR}")
print(f"FIGURES_DIR: {FIGURES_DIR}")

MDPI_FIGURES = "/home/sergioperez/Documents/MDPI_Article/Figures"
import pathlib as _pl
if _pl.Path(MDPI_FIGURES).exists():
    import os as _os
    _os.makedirs(MDPI_FIGURES, exist_ok=True)
    print(f"MDPI export: {MDPI_FIGURES}")
else:
    MDPI_FIGURES = None


## 1. Load benchmark_results.csv

In [6]:
df = pd.read_csv(os.path.join(RESULTS_DIR, "benchmark_results.csv"))
print(f"Runs totales: {len(df)}")
print(f"Models: {sorted(df.model.unique())}")
print(f"Optimizers: {sorted(df.optimizer.unique())}")
print(f"Seeds: {df.seed.nunique()} ({df.seed.min()}–{df.seed.max()})")
print()
print(df.groupby(["model","optimizer"]).size().unstack())


Runs totales: 360
Models: ['GRU', 'LSTM', 'MLP']
Optimizers: ['ABC', 'GA', 'PSO', 'TPE']
Seeds: 30 (0–29)

optimizer  ABC  GA  PSO  TPE
model                       
GRU         30  30   30   30
LSTM        30  30   30   30
MLP         30  30   30   30


## 2. Summary table: mean ± std by model × optimizer

In [7]:
rows = []
for model in MODELS:
    for opt in OPTS:
        sub = df[(df.model==model) & (df.optimizer==opt)]
        rows.append({
            "model": model, "optimizer": opt, "n": len(sub),
            "test_mae_mean": sub.test_mae.mean(), "test_mae_std": sub.test_mae.std(),
            "test_rmse_mean": sub.test_rmse.mean(), "test_rmse_std": sub.test_rmse.std(),
            "test_r2_mean": sub.test_r2.mean(), "test_r2_std": sub.test_r2.std(),
            "daylight_mae_mean": sub.daylight_mae.mean(), "daylight_mae_std": sub.daylight_mae.std(),
            "daylight_r2_mean": sub.daylight_r2.mean(), "daylight_r2_std": sub.daylight_r2.std(),
            "opt_time_min_mean": sub.opt_time_s.mean()/60, "opt_time_min_std": sub.opt_time_s.std()/60,
        })
summary = pd.DataFrame(rows)

# Tabla legible test_mae
pivot = summary.pivot(index="model", columns="optimizer", values=["test_mae_mean","test_mae_std"])
print("=== Test MAE (mean ± std) ===")
for model in MODELS:
    row_str = f"{model:6s}"
    for opt in OPTS:
        m = summary[(summary.model==model)&(summary.optimizer==opt)].iloc[0]
        row_str += f"  {opt}: {m.test_mae_mean:.2f}±{m.test_mae_std:.2f}"
    print(row_str)

summary_path = os.path.join(TABLES_DIR, "benchmark_summary_full.csv")
summary.to_csv(summary_path, index=False)
print(f"\nSaved: {summary_path}")


=== Test MAE (mean ± std) ===
MLP     PSO: 30.36±1.49  GA: 31.27±1.60  ABC: 30.60±1.46  TPE: 30.68±1.52
LSTM    PSO: 29.23±1.52  GA: 28.93±1.73  ABC: 28.55±1.60  TPE: 28.32±1.48
GRU     PSO: 29.56±1.59  GA: 29.40±1.46  ABC: 29.78±1.99  TPE: 29.34±1.84

Saved: /home/jovyan/work/outputs/tables/benchmark_summary_full.csv


## 3. Statistical tests: Friedman + Kruskal-Wallis

In [8]:
print("="*60)
print("FRIEDMAN TEST (efecto del optimizador en test_mae)")
print("="*60)
friedman_results = {}
for model in MODELS:
    groups = [df[(df.model==model)&(df.optimizer==opt)]["test_mae"].values for opt in OPTS]
    stat, p = stats.friedmanchisquare(*groups)
    sig = "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "ns"
    friedman_results[model] = {"stat": float(stat), "p": float(p), "sig": sig}
    print(f"  {model}: chi2={stat:.4f}, p={p:.4f} {sig}")

print()
print("="*60)
print("KRUSKAL-WALLIS TEST")
print("="*60)
kw_results = {}
for model in MODELS:
    groups = [df[(df.model==model)&(df.optimizer==opt)]["test_mae"].values for opt in OPTS]
    stat, p = stats.kruskal(*groups)
    sig = "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "ns"
    kw_results[model] = {"stat": float(stat), "p": float(p), "sig": sig}
    print(f"  {model}: H={stat:.4f}, p={p:.4f} {sig}")


FRIEDMAN TEST (efecto del optimizador en test_mae)
  MLP: chi2=7.5200, p=0.0570 ns
  LSTM: chi2=3.1600, p=0.3676 ns
  GRU: chi2=1.0800, p=0.7819 ns

KRUSKAL-WALLIS TEST
  MLP: H=5.1733, p=0.1595 ns
  LSTM: H=6.6850, p=0.0826 ns
  GRU: H=0.6500, p=0.8849 ns


## 4. Pairwise tests: Mann-Whitney U + Vargha-Delaney A12

In [9]:
def vd_a12(x, y):
    m, n = len(x), len(y)
    r = stats.rankdata(np.concatenate([x, y]))
    rx = r[:m].sum()
    return (rx/m - (m+1)/2) / n

pairwise_rows = []
for model in MODELS:
    print(f"\n{model}:")
    sub = df[df.model==model]
    for o1, o2 in combinations(OPTS, 2):
        x = sub[sub.optimizer==o1]["test_mae"].values
        y = sub[sub.optimizer==o2]["test_mae"].values
        stat, p = stats.mannwhitneyu(x, y, alternative="two-sided")
        a12 = vd_a12(x, y)
        eff = "large" if abs(a12-0.5)>0.21 else "medium" if abs(a12-0.5)>0.14 else "small" if abs(a12-0.5)>0.06 else "negligible"
        sig = "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "ns"
        print(f"  {o1} vs {o2}: p={p:.4f}{sig}, A12={a12:.3f} ({eff})")
        pairwise_rows.append({"model":model,"o1":o1,"o2":o2,"p":p,"A12":a12,"effect":eff,"sig":sig})

pairwise_df = pd.DataFrame(pairwise_rows)
pairwise_df.to_csv(os.path.join(TABLES_DIR, "pairwise_tests.csv"), index=False)
print("\nSaved: pairwise_tests.csv")



MLP:
  PSO vs GA: p=0.0292*, A12=0.336 (medium)
  PSO vs ABC: p=0.6100ns, A12=0.461 (negligible)
  PSO vs TPE: p=0.4290ns, A12=0.440 (negligible)
  GA vs ABC: p=0.1154ns, A12=0.619 (small)
  GA vs TPE: p=0.1580ns, A12=0.607 (small)
  ABC vs TPE: p=0.8187ns, A12=0.482 (negligible)

LSTM:
  PSO vs GA: p=0.4119ns, A12=0.562 (small)
  PSO vs ABC: p=0.0537ns, A12=0.646 (medium)
  PSO vs TPE: p=0.0133*, A12=0.687 (medium)
  GA vs ABC: p=0.3555ns, A12=0.570 (small)
  GA vs TPE: p=0.1907ns, A12=0.599 (small)
  ABC vs TPE: p=0.7506ns, A12=0.524 (negligible)

GRU:
  PSO vs GA: p=0.6100ns, A12=0.539 (negligible)
  PSO vs ABC: p=0.6100ns, A12=0.461 (negligible)
  PSO vs TPE: p=0.8650ns, A12=0.513 (negligible)
  GA vs ABC: p=0.4825ns, A12=0.447 (negligible)
  GA vs TPE: p=0.9587ns, A12=0.504 (negligible)
  ABC vs TPE: p=0.5895ns, A12=0.541 (negligible)

Saved: pairwise_tests.csv


## 5. Figure: Violin plot — Test MAE by model × optimizer

In [ ]:
plt.rcParams.update({
    "font.family":"serif","font.size":9,
    "axes.linewidth":0.8,
    "axes.spines.top":False,"axes.spines.right":False,
})

fig, axes = plt.subplots(1, 3, figsize=(7.0, 3.0), sharey=True)
fig.subplots_adjust(wspace=0.08, left=0.10, right=0.97, top=0.88, bottom=0.18)
rng = np.random.default_rng(42)

for ax, model in zip(axes, MODELS):
    sub = df[df.model==model]
    for i, opt in enumerate(OPTS):
        vals = sub[sub.optimizer==opt]["test_mae"].values
        parts = ax.violinplot(vals, positions=[i], widths=0.6, showmedians=False, showextrema=False)
        for pc in parts["bodies"]:
            pc.set_facecolor(PALETTE[opt]); pc.set_alpha(0.42)
            pc.set_edgecolor(PALETTE[opt]); pc.set_linewidth(0.8)
        q1, med, q3 = np.percentile(vals, [25,50,75])
        ax.plot([i,i],[q1,q3], color=PALETTE[opt], lw=2.0, solid_capstyle="round")
        ax.scatter([i],[med], color="white", edgecolors=PALETTE[opt], s=28, zorder=5, linewidth=1.2)
        jit = rng.uniform(-0.12, 0.12, len(vals))
        ax.scatter(i+jit, vals, color=PALETTE[opt], alpha=0.55, s=14, zorder=4)
    ax.set_title(model, fontsize=10, fontweight="bold", pad=4)
    ax.set_xticks(range(len(OPTS))); ax.set_xticklabels(OPTS, fontsize=8.5)
    ax.set_xlim(-0.6, len(OPTS)-0.4)
    ax.grid(axis="y", lw=0.4, color="#cccccc", zorder=0)

axes[0].set_ylabel("Test MAE (W/m²)", fontsize=9)
patches = [mpatches.Patch(facecolor=PALETTE[o], alpha=0.7, label=o) for o in OPTS]
fig.legend(handles=patches, ncol=4, loc="upper center",
           bbox_to_anchor=(0.53, 1.01), fontsize=8.5, frameon=False)

for ext in ("pdf","png"):
    fig.savefig(os.path.join(FIGURES_DIR, f"violin_mae.{ext}"), dpi=300, bbox_inches="tight")
    if MDPI_FIGURES:
        fig.savefig(os.path.join(MDPI_FIGURES, f"fa_violin_mae.{ext}"), dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved violin_mae.pdf/.png")


## 6. Figure: Heatmap — Mean Test MAE by model × optimizer

In [ ]:
mean_tbl = df.groupby(["model","optimizer"])["test_mae"].mean().unstack()[OPTS].loc[MODELS]
std_tbl  = df.groupby(["model","optimizer"])["test_mae"].std().unstack()[OPTS].loc[MODELS]

fig, ax = plt.subplots(figsize=(5.5, 2.8))
vmin, vmax = mean_tbl.values.min(), mean_tbl.values.max()
im = ax.imshow(mean_tbl.values, cmap="RdYlGn_r", aspect="auto",
               vmin=vmin*0.99, vmax=vmax*1.01)

ax.set_xticks(range(len(OPTS)));   ax.set_xticklabels(OPTS, fontsize=9)
ax.set_yticks(range(len(MODELS))); ax.set_yticklabels(MODELS, fontsize=9)

for r, model in enumerate(MODELS):
    for c, opt in enumerate(OPTS):
        m = mean_tbl.loc[model, opt]; s = std_tbl.loc[model, opt]
        ax.text(c, r, f"{m:.2f}\n±{s:.2f}", ha="center", va="center",
                fontsize=7.5, color="black")

plt.colorbar(im, ax=ax, label="Mean Test MAE (W/m²)", pad=0.02)
ax.set_title("Mean Test MAE by Model and Optimizer", fontsize=10, pad=8)
fig.tight_layout()

for ext in ("pdf","png"):
    fig.savefig(os.path.join(FIGURES_DIR, f"heatmap_mae.{ext}"), dpi=300, bbox_inches="tight")
    if MDPI_FIGURES:
        fig.savefig(os.path.join(MDPI_FIGURES, f"fb_heatmap_mae.{ext}"), dpi=300, bbox_inches="tight")
plt.close()
print("Saved heatmap_mae.pdf/.png")


## 7. Article table (LaTeX-ready)

In [12]:
lines = []
lines.append(r"\begin{tabular}{llcccc}")
lines.append(r"\toprule")
lines.append(r"Model & Metric & PSO & GA & ABC & TPE \\")
lines.append(r"\midrule")

for model in MODELS:
    first = True
    for metric, label in [("test_mae","MAE (W/m²)"),("test_rmse","RMSE (W/m²)"),("test_r2","R²")]:
        row_parts = [f"\\multirow{{3}}{{*}}{{{model}}}" if first else "", label]
        first = False
        for opt in OPTS:
            m = df[(df.model==model)&(df.optimizer==opt)][metric].mean()
            s = df[(df.model==model)&(df.optimizer==opt)][metric].std()
            row_parts.append(f"{m:.3f} $\\pm$ {s:.3f}")
        lines.append(" & ".join(row_parts) + r" \\")
    lines.append(r"\midrule")

lines.append(r"\bottomrule")
lines.append(r"\end{tabular}")

latex_str = "\n".join(lines)
latex_path = os.path.join(TABLES_DIR, "article_metrics_table.tex")
with open(latex_path, "w") as f:
    f.write(latex_str)
print(latex_str)
print(f"\nSaved: {latex_path}")


\begin{tabular}{llcccc}
\toprule
Model & Metric & PSO & GA & ABC & TPE \\
\midrule
\multirow{3}{*}{MLP} & MAE (W/m²) & 30.362 $\pm$ 1.491 & 31.269 $\pm$ 1.597 & 30.600 $\pm$ 1.460 & 30.677 $\pm$ 1.520 \\
 & RMSE (W/m²) & 55.868 $\pm$ 1.915 & 56.272 $\pm$ 2.049 & 55.525 $\pm$ 1.365 & 56.505 $\pm$ 2.181 \\
 & R² & 0.970 $\pm$ 0.002 & 0.969 $\pm$ 0.002 & 0.970 $\pm$ 0.001 & 0.969 $\pm$ 0.002 \\
\midrule
\multirow{3}{*}{LSTM} & MAE (W/m²) & 29.230 $\pm$ 1.520 & 28.930 $\pm$ 1.729 & 28.550 $\pm$ 1.599 & 28.320 $\pm$ 1.476 \\
 & RMSE (W/m²) & 54.036 $\pm$ 2.101 & 54.097 $\pm$ 2.479 & 53.355 $\pm$ 1.943 & 53.141 $\pm$ 1.920 \\
 & R² & 0.971 $\pm$ 0.002 & 0.971 $\pm$ 0.003 & 0.972 $\pm$ 0.002 & 0.972 $\pm$ 0.002 \\
\midrule
\multirow{3}{*}{GRU} & MAE (W/m²) & 29.558 $\pm$ 1.593 & 29.403 $\pm$ 1.456 & 29.776 $\pm$ 1.992 & 29.340 $\pm$ 1.841 \\
 & RMSE (W/m²) & 54.094 $\pm$ 1.948 & 53.969 $\pm$ 1.986 & 54.042 $\pm$ 2.359 & 53.848 $\pm$ 2.014 \\
 & R² & 0.971 $\pm$ 0.002 & 0.972 $\pm$ 0.002 & 0.9

## 8. Final summary

In [13]:
print("="*60)
print("RESUMEN FINAL DEL BENCHMARK")
print("="*60)
print(f"Total runs: {len(df)}")
print(f"Seeds: {df.seed.nunique()}")
print()
print("Mejor combinación (test_mae mínimo):")
best = df.loc[df.test_mae.idxmin()]
print(f"  {best.model} + {best.optimizer}, seed {best.seed}: {best.test_mae:.3f} W/m²")
print()
print("Ranking de modelos (mean test_mae, todos los optimizadores):")
print(df.groupby("model")["test_mae"].mean().sort_values().to_string())
print()
print("Ranking de optimizadores (mean test_mae, todos los modelos):")
print(df.groupby("optimizer")["test_mae"].mean().sort_values().to_string())
print()
print("Conclusión estadística:")
for model, res in friedman_results.items():
    print(f"  {model}: Friedman p={res["p"]:.4f} ({res["sig"]}) — optimizadores {"equivalentes" if res["sig"]=="ns" else "significativamente distintos"}")


RESUMEN FINAL DEL BENCHMARK
Total runs: 360
Seeds: 30

Mejor combinación (test_mae mínimo):
  LSTM + TPE, seed 19: 25.779 W/m²

Ranking de modelos (mean test_mae, todos los optimizadores):
model
LSTM    28.757427
GRU     29.519326
MLP     30.727167

Ranking de optimizadores (mean test_mae, todos los modelos):
optimizer
TPE    29.445744
ABC    29.641898
PSO    29.716770
GA     29.867481

Conclusión estadística:
  MLP: Friedman p=0.0570 (ns) — optimizadores equivalentes
  LSTM: Friedman p=0.3676 (ns) — optimizadores equivalentes
  GRU: Friedman p=0.7819 (ns) — optimizadores equivalentes
